In [ ]:
import sys, os, pickle, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
from spk_lfp_cluster_comp_analysis import *
from config import SPE1_PICKLE_ROOT, PRIORITY_CELLS, CELL_IDS

warnings.filterwarnings("ignore")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
SLIDING_DIR  = os.path.join(SPE1_PICKLE_ROOT, "lfp_spk_group_pickles")
PREPOST_DIR  = os.path.join(SPE1_PICKLE_ROOT, "prepost_specparam_pickles")
CLUSTER_DIR  = os.path.join(SPE1_PICKLE_ROOT, "cluster_pickles")

# ── Cell subsets ───────────────────────────────────────────────────────────────
subsets = get_cell_subsets(PRIORITY_CELLS, CELL_IDS)
# subsets["priority"] — priority cells only
# subsets["np"]       — non-priority cells
# subsets["all"]      — all cells combined
print(f"Priority: {len(subsets['priority'])} cells")
print(f"NP:       {len(subsets['np'])} cells")
print(f"All:      {len(subsets['all'])} cells")
# Spike feature colour palette (consistent across all notebooks)
feature_shades = {
    'peak_amp_cluster':         '#8c564b',
    'peak_sharpness_cluster':   '#a06d62',
    'peak_width_cluster':       '#b38479',
    'exp_lambda_cluster':       '#c561a8',
    'inflection_time_cluster':  '#9b59b6',
    'exp_const_cluster':        '#d7aee0',
    'log_isi_cluster':          '#7f7f7f',
    'spk_times_ms_cluster':     '#b0b0b0',
}


## 1. Load Data

In [ ]:
df_prepost = compile_prepost_stats(PREPOST_DIR)
print(f'Loaded {df_prepost["cell_id"].nunique()} cells, {len(df_prepost)} tests')

In [ ]:
SUBSET_LABELS = [
    ("Priority cells",      subsets["priority"]),
    ("Non-priority cells",  subsets["np"]),
    ("All cells combined",  subsets["all"]),
]

## 2. Effect Sizes — Pre & Post Windows

Forest plots of Hedges' g (between-group) and Cohen's d_z (within-group) with 95% bootstrap CI. Filled = significant after BH-FDR correction.

In [ ]:
for label, cell_ids in SUBSET_LABELS:
    print(f"\n{'='*60}\n{label}\n{'='*60}")
    df_sub = df_prepost[df_prepost["cell_id"].isin(cell_ids)]
    plot_prepost_effect_sizes(df_sub, feature_shades=feature_shades, subset_label=label)

## 3. Significant Yield Heatmap

Fraction of cells with p_fdr < 0.05 for each spike × LFP feature × window.

In [ ]:
for label, cell_ids in SUBSET_LABELS:
    print(f"\n{'='*60}\n{label}\n{'='*60}")
    df_sub = df_prepost[df_prepost["cell_id"].isin(cell_ids)]
    plot_prepost_yield(df_sub, subset_label=label)

## 4. Interaction Effects

Does the A−B cluster difference **change** from pre to post spike?
This is the most scientifically interesting question.

In [ ]:
for label, cell_ids in SUBSET_LABELS:
    print(f"\n{'='*60}\n{label}\n{'='*60}")
    df_sub = df_prepost[df_prepost["cell_id"].isin(cell_ids)]
    df_int = df_sub[df_sub["window"] == "interaction"]
    plot_prepost_effect_sizes(df_int, feature_shades=feature_shades,
                              subset_label=f"{label} — Interaction")

## 5. Priority vs Non-Priority Comparison

Do priority cells show stronger pre/post modulation?

In [ ]:
df_pri = df_prepost[df_prepost["cell_id"].isin(subsets["priority"])].copy()
df_np  = df_prepost[df_prepost["cell_id"].isin(subsets["np"])].copy()
df_pri["subset"] = "Priority"
df_np["subset"]  = "NP"
df_compare = pd.concat([df_pri, df_np])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, win, es_col, es_label in [
    (axes[0], "pre",    "hedges_g",  "Hedges' g [pre]"),
    (axes[1], "within", "cohens_dz", "Cohen's d_z [pre→post]"),
]:
    df_w = df_compare[df_compare["window"] == win].dropna(subset=[es_col])
    if df_w.empty: continue
    sns.boxplot(data=df_w, x="lfp_feature", y=es_col, hue="subset",
                ax=ax, palette={"Priority": "#0072B2", "NP": "#D55E00"})
    ax.axhline(0, color="gray", ls="--", lw=0.8)
    ax.set_xlabel("LFP Feature"); ax.set_ylabel(es_label)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)
    sns.despine(ax=ax)
plt.suptitle("Priority vs Non-Priority: Pre/Post Effect Sizes", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()